In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
BASE_URL = "https://en.numista.com"

In [3]:
def get_page_html(url: str) -> str:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1600,1400")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        driver.get(url)
        time.sleep(2)
        html = driver.page_source
    finally:
        driver.quit()

    return html

In [4]:
def scrape_note_links_all_pages(first_page_url: str, issuer_name: str) -> list:
    page = 1
    rows = []

    while True:
        url = first_page_url.replace("-1.html", f"-{page}.html")
        print(f"Scraping {issuer_name} - page {page}")

        html = get_page_html(url)
        soup = BeautifulSoup(html, "html.parser")

        blocks = soup.select("div.resultat_recherche div.description_piece")

        if not blocks:
            break

        for block in blocks:
            a = block.select_one("strong a")
            if not a:
                continue

            href = a.get("href")  # e.g. /247092
            note_url = BASE_URL + href
            title = a.get_text(" ", strip=True)

            rows.append({
                "issuer_name": issuer_name,
                "issuer_page": url,
                "note_title": title,
                "note_url": note_url
            })

        page += 1

    return rows

In [5]:
# -------------------------
# MAIN
# -------------------------

# Load issuers
df_issuers = pd.read_csv("numista_issuers_tree.csv")

# Keep only main catalogue pages (the ones ending -1.html)
df_issuers = df_issuers[df_issuers["url"].str.contains("-1.html", na=False)]

In [6]:
# TEMP FILTER FOR TESTING (REMOVE LATER)
df_issuers = df_issuers[df_issuers["name"] == "Slovakia"]

In [ ]:
df_issuers.head()

,name,url,relative_url,alt_names,path,depth,li_classes
6598,Slovakia,https://en.numista.com/catalogue/slovaquie-1.html,/catalogue/slovaquie-1.html,Slovak Republic Slovensko Slovenská republika ...,Slovakia,0,tag_europe tag_modern


In [8]:
all_rows = []

for _, row in df_issuers.iterrows():
    issuer_name = row["name"]
    issuer_url = row["url"]

    rows = scrape_note_links_all_pages(issuer_url, issuer_name)
    all_rows.extend(rows)

df_notes = pd.DataFrame(all_rows)

print(df_notes.head())
print("Total notes scraped:", len(df_notes))

df_notes.to_csv("numista_note_links.csv", index=False)

Scraping Slovakia - page 1
Scraping Slovakia - page 2
  issuer_name                                        issuer_page  \
0    Slovakia  https://en.numista.com/catalogue/slovaquie-1.html   
1    Slovakia  https://en.numista.com/catalogue/slovaquie-1.html   
2    Slovakia  https://en.numista.com/catalogue/slovaquie-1.html   
3    Slovakia  https://en.numista.com/catalogue/slovaquie-1.html   
4    Slovakia  https://en.numista.com/catalogue/slovaquie-1.html   

                       note_title                       note_url  
0                 5 Halierov 1942   https://en.numista.com/14333  
1  5 Halierov (Trial Strike) 1942  https://en.numista.com/447485  
2           10 Halierov 1939-1942    https://en.numista.com/9945  
3      10 Halierov (Pattern) 1943  https://en.numista.com/430159  
4      10 Halierov (Pattern) 1943   https://en.numista.com/91623  
Total notes scraped: 50
